# Module 09: RL for Object Detection
## Notebook 4: Multi-Object Detection with Sequential RL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_09_RL_For_Object_Detection/04_Multi_Object_Detection/multi_object_detection.ipynb)

---

### Learning Objectives

1. Formulate multi-object detection as a sequential decision-making problem
2. Design state representations that encode already-detected objects
3. Implement a DQN agent that detects objects one at a time without duplicates
4. Visualize sequential detection order and coverage
5. Analyze performance scaling with number of objects

---

**Key Idea:** The agent detects objects sequentially — one at a time — maintaining a memory of previously detected objects to avoid duplicates and to know when all objects have been found.

In [ ]:
!pip install torch torchvision numpy matplotlib scikit-image

In [ ]:
# Download REAL open-source dataset for object detection (TINY - under 200MB)
import torchvision
import numpy as np

# MNIST: Real handwritten digits for detection scenes
mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True)
print(f"✅ MNIST: {len(mnist)} real handwritten digits (11MB)")

# CIFAR-10: Real photographs
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10: {len(cifar10)} real photos (170MB)")

# Create detection dataset from MNIST (place digits on canvas with known bounding boxes)
def create_detection_dataset(n_scenes=200, canvas_size=84, max_objects=3):
    """Create real detection scenes using MNIST digits on canvas."""
    scenes = []
    for _ in range(n_scenes):
        canvas = np.zeros((canvas_size, canvas_size), dtype=np.uint8)
        boxes, labels = [], []
        n_obj = np.random.randint(1, max_objects + 1)
        for _ in range(n_obj):
            idx = np.random.randint(len(mnist))
            digit_img = np.array(mnist[idx][0])
            label = mnist[idx][1]
            x = np.random.randint(0, canvas_size - 28)
            y = np.random.randint(0, canvas_size - 28)
            canvas[y:y+28, x:x+28] = np.maximum(canvas[y:y+28, x:x+28], digit_img)
            boxes.append([x, y, x+28, y+28])
            labels.append(label)
        scenes.append({'image': canvas, 'boxes': boxes, 'labels': labels})
    return scenes

detection_data = create_detection_dataset(200)
print(f"✅ Created {len(detection_data)} detection scenes with REAL digit images")
print(f"   Each scene has 1-3 objects with ground-truth bounding boxes")
print(f"   Total download: ~181MB (MNIST + CIFAR-10)")

## Deep Derivation: Multi-Object Detection with Sequential RL

### Step 1: Multi-Object MDP Extension
Extend single-object localization to $K$ objects. The state now tracks which objects have been found:
$$s_t = (\phi(I), B_t, \mathbf{d}_t)$$

where $\mathbf{d}_t \in \{0,1\}^K$ is a detection indicator vector: $d_t^k = 1$ if object $k$ has been detected (IoU with some prediction $\geq \tau$).

**Joint action space:** $\mathcal{A} = \mathcal{A}_{\text{move}} \cup \{$trigger, next\_object$\}$

The "next_object" action resets the bounding box and begins searching for an undetected object.

### Step 2: Hungarian Algorithm for Assignment
To match $M$ predictions to $N$ ground truths, solve the assignment problem:
$$\sigma^* = \arg\min_{\sigma \in \mathfrak{S}_N} \sum_{i=1}^N \mathcal{L}_{\text{match}}(y_i, \hat{y}_{\sigma(i)})$$

where $\mathfrak{S}_N$ is the set of permutations and:
$$\mathcal{L}_{\text{match}} = -\mathbb{1}_{c_i \neq \varnothing}\hat{p}_{\sigma(i)}(c_i) + \mathbb{1}_{c_i \neq \varnothing}\mathcal{L}_{\text{box}}(b_i, \hat{b}_{\sigma(i)})$$

**Complexity:** The Hungarian algorithm runs in $O(N^3)$.

**Derivation:** This is a linear assignment problem. Construct cost matrix $C_{ij} = \mathcal{L}_{\text{match}}(y_i, \hat{y}_j)$. The algorithm iteratively finds augmenting paths in the bipartite graph to minimize total cost. The dual problem provides optimality certificates:

$$\min \sum_{ij} C_{ij} x_{ij} \quad \text{s.t.} \quad \sum_j x_{ij} = 1, \quad \sum_i x_{ij} = 1, \quad x_{ij} \geq 0$$

By total unimodularity of the constraint matrix, the LP relaxation has integral optimal solutions. $\blacksquare$

### Step 3: Non-Maximum Suppression (NMS) as Post-Processing
Given detected boxes $\{(B_i, s_i)\}$ with confidence scores $s_i$:

1. Sort by score: $s_{\pi(1)} \geq s_{\pi(2)} \geq \cdots$
2. For each box $B_{\pi(i)}$ in order:
   - Suppress all $B_{\pi(j)}$ where $j > i$ and $\text{IoU}(B_{\pi(i)}, B_{\pi(j)}) > \tau_{\text{NMS}}$

**Soft-NMS (continuous relaxation):**
$$s_j \leftarrow s_j \cdot e^{-\frac{\text{IoU}(B_i, B_j)^2}{\sigma}}$$

**Derivation:** Hard NMS is a greedy approximation to the maximum weight independent set problem on the IoU graph $G = (V, E)$ where $V$ = boxes and $(i,j) \in E$ iff $\text{IoU}(B_i, B_j) > \tau$. This problem is NP-hard in general, but the geometric structure of bounding boxes makes greedy perform well. $\blacksquare$

### Step 4: Multi-Object Reward Design
$$R_t = \underbrace{\sum_{k=1}^K \Delta\text{IoU}_k \cdot (1 - d_t^k)}_{\text{localization improvement}} + \underbrace{\lambda_1 \sum_{k} \mathbb{1}[\text{newly detected } k]}_{\text{detection bonus}} - \underbrace{\lambda_2 \cdot \mathbb{1}[\text{duplicate detection}]}_{\text{redundancy penalty}}$$

**Why the $(1 - d_t^k)$ mask?** Without it, the agent could repeatedly refine already-detected objects instead of finding new ones. The mask redirects attention to undetected objects.

**Derivation of optimal $\lambda_1, \lambda_2$:** Consider the ratio of detection bonus to average IoU improvement per step. If $\Delta\text{IoU}_{\text{avg}} \approx 0.05$ and we want the agent to prefer detecting a new object over improving an existing detection by one step:
$$\lambda_1 > \Delta\text{IoU}_{\text{avg}} \implies \lambda_1 > 0.05$$

Similarly, $\lambda_2$ must exceed $\lambda_1$ to prevent duplicate detections:
$$\lambda_2 > \lambda_1$$

### Step 5: Mean Average Precision (mAP) Derivation
For each class $c$, compute precision-recall curve. Precision at recall $r$:
$$p(r) = \max_{\tilde{r} \geq r} p(\tilde{r})$$

Average Precision for class $c$:
$$\text{AP}_c = \int_0^1 p(r)\, dr \approx \sum_{k=1}^{n} (r_k - r_{k-1}) \cdot p(r_k)$$

Mean Average Precision:
$$\text{mAP} = \frac{1}{|\mathcal{C}|} \sum_{c \in \mathcal{C}} \text{AP}_c$$

**mAP@[0.5:0.95]:** Average over IoU thresholds $\tau \in \{0.5, 0.55, \ldots, 0.95\}$:
$$\text{mAP} = \frac{1}{10}\sum_{i=0}^{9} \text{mAP}@(0.5 + 0.05i)$$

### Step 6: Combinatorial Complexity Analysis
With $K$ objects, the number of possible detection orderings is $K!$. For $K = 10$, this is $3,628,800$ orderings.

**RL addresses this:** Instead of enumerating all orderings, the policy learns to prioritize:
$$\pi^*(a | s_t) \propto \exp\left(\frac{Q(s_t, a)}{\tau}\right)$$

The Q-function implicitly encodes which undetected object is easiest to find next, naturally learning a near-optimal detection ordering without combinatorial search. $\blacksquare$

### Step 7: Convergence Guarantee for Multi-Object Q-Learning
**Theorem:** Under the standard conditions (all state-action pairs visited infinitely often, learning rate satisfying Robbins-Monro conditions), multi-object Q-learning converges to $Q^*$ if the environment is ergodic.

**Proof sketch:** The multi-object MDP is a valid MDP (state includes detection history $\mathbf{d}_t$). The augmented state space has $|\mathcal{S}| \times 2^K$ states. Standard Q-learning convergence (Watkins & Dayan, 1992) applies to this augmented MDP. The convergence rate is $O(1/\sqrt{t})$ for the tabular case. $\blacksquare$

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import deque
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

np.random.seed(42)
torch.manual_seed(42)

## 1. Mathematical Formulation: MDP for Sequential Multi-Object Detection

### Problem Setting

Given an image $\mathbf{I}$ containing $K$ objects $\{o_1, \ldots, o_K\}$ at unknown locations, the agent must sequentially detect all objects while avoiding duplicate detections.

### MDP Definition $(\mathcal{S}, \mathcal{A}, \mathcal{T}, \mathcal{R}, \gamma)$

### State Space $\mathcal{S}$

The state encodes both the image and detection history:

$$s_t = \left(\phi(\mathbf{I}),\; \mathbf{M}_t,\; n_t\right)$$

where:
- $\phi(\mathbf{I}) \in \mathbb{R}^{C \times H' \times W'}$ is a feature representation of the image
- $\mathbf{M}_t \in \{0,1\}^{H \times W}$ is a **detection mask** marking already-detected regions:
  $$\mathbf{M}_t(i,j) = \begin{cases} 1 & \text{if pixel }(i,j)\text{ is in a detected box} \\ 0 & \text{otherwise} \end{cases}$$
- $n_t \in \mathbb{N}$ is the count of objects detected so far

### Action Space $\mathcal{A}$

We discretize the image into an $N \times N$ grid. The action selects a grid cell to propose a detection:

$$\mathcal{A} = \{(i, j) : i \in \{0,\ldots,N-1\}, j \in \{0,\ldots,N-1\}\} \cup \{\text{stop}\}$$

Total actions: $N^2 + 1$

Action $(i,j)$ proposes a bounding box centered at grid cell $(i,j)$:
$$\mathbf{b}_{(i,j)} = \left(j \cdot \frac{W}{N} + \frac{W}{2N} - \frac{w_d}{2},\; i \cdot \frac{H}{N} + \frac{H}{2N} - \frac{h_d}{2},\; w_d,\; h_d\right)$$

where $(w_d, h_d)$ is a default object size.

### Reward Function $\mathcal{R}$

$$R(s_t, a_t) = \begin{cases}
+1 & \text{if } a_t = (i,j) \text{ and } \exists k: \text{IoU}(\mathbf{b}_{(i,j)}, o_k) \geq 0.5 \text{ and } o_k \text{ not yet detected} \\
-1 & \text{if } a_t = (i,j) \text{ and detection is duplicate (already detected)} \\
-0.5 & \text{if } a_t = (i,j) \text{ and no object at that location} \\
+3 & \text{if } a_t = \text{stop and all objects detected} \\
-2 & \text{if } a_t = \text{stop and objects remain} \\
\end{cases}$$

### Non-Maximum Suppression via State

The detection mask $\mathbf{M}_t$ serves as an implicit NMS mechanism. When the agent detects an object, the mask is updated:

$$\mathbf{M}_{t+1}(i,j) = \max\left(\mathbf{M}_t(i,j),\; \mathbb{1}_{(i,j) \in \mathbf{b}_{a_t}}\right)$$

The agent learns to avoid proposing detections in already-masked regions.

### Optimal Policy

The optimal policy maximizes:

$$J(\pi) = \mathbb{E}_\pi\left[\sum_{t=0}^T \gamma^t R(s_t, a_t)\right]$$

This is equivalent to finding all objects in minimum steps:

$$\pi^* = \arg\max_\pi \mathbb{E}_\pi\left[\text{Recall} - \lambda \cdot T\right]$$

where Recall $= \frac{\text{\# correctly detected}}{K}$ and $T$ is the number of steps.

## 2. Multi-Object Detection Environment

In [ ]:
class MultiObjectEnv:
    """Environment with multiple objects for sequential detection."""
    
    def __init__(self, image_size=64, grid_size=8, min_objects=2, max_objects=5,
                 object_size=10, max_steps=15):
        self.image_size = image_size
        self.grid_size = grid_size
        self.min_objects = min_objects
        self.max_objects = max_objects
        self.object_size = object_size
        self.max_steps = max_steps
        self.cell_size = image_size // grid_size
        
        self.n_actions = grid_size * grid_size + 1  # grid cells + stop
        self.action_names = [f'cell({i},{j})' for i in range(grid_size) 
                            for j in range(grid_size)] + ['stop']
    
    def _generate_scene(self):
        """Generate image with multiple non-overlapping objects."""
        image = np.random.rand(self.image_size, self.image_size) * 0.2 + 0.05
        
        n_objects = np.random.randint(self.min_objects, self.max_objects + 1)
        self.objects = []
        
        shapes = ['circle', 'square', 'triangle']
        colors_val = [0.7, 0.8, 0.9, 0.95]
        
        for _ in range(n_objects):
            for attempt in range(50):
                size = self.object_size + np.random.randint(-2, 3)
                x = np.random.randint(2, self.image_size - size - 2)
                y = np.random.randint(2, self.image_size - size - 2)
                
                # Check no overlap with existing objects
                overlap = False
                for obj in self.objects:
                    ox, oy, ow, oh = obj['bbox']
                    if (x < ox + ow + 3 and x + size > ox - 3 and
                        y < oy + oh + 3 and y + size > oy - 3):
                        overlap = True
                        break
                
                if not overlap:
                    shape = np.random.choice(shapes)
                    color = np.random.choice(colors_val)
                    
                    # Draw shape
                    if shape == 'square':
                        image[y:y+size, x:x+size] = color
                    elif shape == 'circle':
                        cy, cx = y + size//2, x + size//2
                        r = size // 2
                        yy, xx = np.ogrid[:self.image_size, :self.image_size]
                        mask = (xx - cx)**2 + (yy - cy)**2 <= r**2
                        image[mask] = color
                    else:  # triangle
                        for row in range(size):
                            half_w = int(size * (row / size) / 2)
                            cx = x + size // 2
                            left = max(cx - half_w, 0)
                            right = min(cx + half_w + 1, self.image_size)
                            yr = min(y + row, self.image_size - 1)
                            image[yr, left:right] = color
                    
                    self.objects.append({
                        'bbox': (x, y, size, size),
                        'shape': shape,
                        'detected': False
                    })
                    break
        
        return image
    
    def _action_to_bbox(self, action):
        """Convert grid action to bounding box."""
        if action >= self.grid_size * self.grid_size:
            return None
        
        row = action // self.grid_size
        col = action % self.grid_size
        
        cx = col * self.cell_size + self.cell_size // 2
        cy = row * self.cell_size + self.cell_size // 2
        
        half_size = self.object_size // 2 + 2
        x = max(0, cx - half_size)
        y = max(0, cy - half_size)
        w = min(self.object_size + 4, self.image_size - x)
        h = min(self.object_size + 4, self.image_size - y)
        
        return (x, y, w, h)
    
    def _compute_iou(self, box1, box2):
        """Compute IoU between two boxes."""
        x1, y1, w1, h1 = box1
        x2, y2, w2, h2 = box2
        
        ix1 = max(x1, x2)
        iy1 = max(y1, y2)
        ix2 = min(x1 + w1, x2 + w2)
        iy2 = min(y1 + h1, y2 + h2)
        
        if ix2 <= ix1 or iy2 <= iy1:
            return 0.0
        
        inter = (ix2 - ix1) * (iy2 - iy1)
        union = w1 * h1 + w2 * h2 - inter
        return inter / max(union, 1e-6)
    
    def _get_state(self):
        """Get state: downsampled image + detection mask + detection count."""
        from skimage.transform import resize
        
        # Downsampled image features
        img_small = resize(self.image, (self.grid_size, self.grid_size), anti_aliasing=True)
        
        # Detection mask (which grid cells have been detected)
        mask_small = resize(self.detection_mask.astype(float), 
                           (self.grid_size, self.grid_size), anti_aliasing=True)
        
        # Combine: image + mask + count
        state = np.concatenate([
            img_small.flatten(),
            mask_small.flatten(),
            [self.n_detected / self.max_objects]
        ])
        
        return state.astype(np.float32)
    
    def reset(self):
        """Reset environment with new scene."""
        self.image = self._generate_scene()
        self.detection_mask = np.zeros((self.image_size, self.image_size))
        self.n_detected = 0
        self.steps = 0
        self.detections = []
        
        return self._get_state()
    
    def step(self, action):
        """Execute detection action."""
        self.steps += 1
        
        # Stop action
        if action == self.grid_size * self.grid_size:
            all_found = all(obj['detected'] for obj in self.objects)
            reward = 3.0 if all_found else -2.0
            info = {
                'n_detected': self.n_detected,
                'n_objects': len(self.objects),
                'recall': self.n_detected / max(len(self.objects), 1),
                'steps': self.steps,
                'stopped': True
            }
            return self._get_state(), reward, True, info
        
        # Propose detection
        bbox = self._action_to_bbox(action)
        
        # Check if it matches any undetected object
        best_iou = 0
        best_obj_idx = -1
        is_duplicate = False
        
        for idx, obj in enumerate(self.objects):
            iou = self._compute_iou(bbox, obj['bbox'])
            if iou > best_iou:
                best_iou = iou
                best_obj_idx = idx
                if obj['detected']:
                    is_duplicate = True
        
        if best_iou >= 0.3 and not is_duplicate:
            # New object detected
            reward = 1.0
            self.objects[best_obj_idx]['detected'] = True
            self.n_detected += 1
            # Update detection mask
            ox, oy, ow, oh = self.objects[best_obj_idx]['bbox']
            y1, y2 = max(0, oy), min(self.image_size, oy + oh)
            x1, x2 = max(0, ox), min(self.image_size, ox + ow)
            self.detection_mask[y1:y2, x1:x2] = 1.0
            self.detections.append({
                'bbox': bbox, 'step': self.steps, 'type': 'correct'
            })
        elif is_duplicate:
            reward = -1.0
            self.detections.append({
                'bbox': bbox, 'step': self.steps, 'type': 'duplicate'
            })
        else:
            reward = -0.5
            self.detections.append({
                'bbox': bbox, 'step': self.steps, 'type': 'false_positive'
            })
        
        # Check termination
        all_found = all(obj['detected'] for obj in self.objects)
        done = self.steps >= self.max_steps or all_found
        
        if all_found and not done:
            reward += 2.0  # Bonus for finding all
        
        info = {
            'n_detected': self.n_detected,
            'n_objects': len(self.objects),
            'recall': self.n_detected / max(len(self.objects), 1),
            'steps': self.steps,
            'stopped': False
        }
        
        return self._get_state(), reward, done, info

# Test
env = MultiObjectEnv()
state = env.reset()
print(f"State dim: {state.shape[0]} ({env.grid_size}x{env.grid_size} image + mask + count)")
print(f"Actions: {env.n_actions} ({env.grid_size}x{env.grid_size} grid + stop)")
print(f"Objects in scene: {len(env.objects)}")
for i, obj in enumerate(env.objects):
    print(f"  Object {i+1}: {obj['shape']} at {obj['bbox']}")

## 3. DQN Agent for Multi-Object Detection

In [ ]:
class MultiObjDQN(nn.Module):
    def __init__(self, state_dim, n_actions):
        super(MultiObjDQN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, n_actions)
        )
    
    def forward(self, x):
        return self.net(x)


class MultiObjAgent:
    def __init__(self, state_dim, n_actions, lr=1e-3, gamma=0.95,
                 eps_start=1.0, eps_end=0.05, eps_decay=0.996):
        self.n_actions = n_actions
        self.gamma = gamma
        self.epsilon = eps_start
        self.eps_end = eps_end
        self.eps_decay = eps_decay
        
        self.q_net = MultiObjDQN(state_dim, n_actions).to(device)
        self.target_net = MultiObjDQN(state_dim, n_actions).to(device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer = deque(maxlen=15000)
        self.batch_size = 64
        self.steps = 0
    
    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            return self.q_net(state_t).argmax(1).item()
    
    def store(self, s, a, r, s_next, done):
        self.buffer.append((s, a, r, s_next, done))
    
    def update(self):
        if len(self.buffer) < self.batch_size:
            return 0.0
        
        batch = random.sample(self.buffer, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        
        s = torch.FloatTensor(np.array(states)).to(device)
        a = torch.LongTensor(actions).to(device)
        r = torch.FloatTensor(rewards).to(device)
        s_next = torch.FloatTensor(np.array(next_states)).to(device)
        d = torch.FloatTensor(dones).to(device)
        
        q_val = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        
        with torch.no_grad():
            next_q = self.target_net(s_next).max(1)[0]
            target = r + self.gamma * next_q * (1 - d)
        
        loss = nn.SmoothL1Loss()(q_val, target)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), 5.0)
        self.optimizer.step()
        
        self.steps += 1
        if self.steps % 150 == 0:
            self.target_net.load_state_dict(self.q_net.state_dict())
        
        self.epsilon = max(self.eps_end, self.epsilon * self.eps_decay)
        return loss.item()

state_dim = env.grid_size * env.grid_size * 2 + 1  # 129
agent = MultiObjAgent(state_dim=state_dim, n_actions=env.n_actions)
print(f"State dim: {state_dim}")
print(f"Action space: {env.n_actions}")
print(agent.q_net)

## 4. Training

In [ ]:
def train_multi_obj(agent, env, n_episodes=700):
    """Train multi-object detection agent."""
    rewards_hist = []
    recall_hist = []
    duplicates_hist = []
    steps_hist = []
    
    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0
        done = False
        n_duplicates = 0
        
        while not done:
            action = agent.select_action(state)
            next_state, reward, done, info = env.step(action)
            agent.store(state, action, reward, next_state, float(done))
            agent.update()
            
            if reward == -1.0:  # duplicate
                n_duplicates += 1
            
            state = next_state
            total_reward += reward
        
        rewards_hist.append(total_reward)
        recall_hist.append(info['recall'])
        duplicates_hist.append(n_duplicates)
        steps_hist.append(info['steps'])
        
        if (ep + 1) % 100 == 0:
            avg_recall = np.mean(recall_hist[-100:])
            avg_dup = np.mean(duplicates_hist[-100:])
            avg_reward = np.mean(rewards_hist[-100:])
            print(f"Ep {ep+1:4d} | Recall: {avg_recall:.2%} | Duplicates: {avg_dup:.2f} | "
                  f"Reward: {avg_reward:.2f} | Eps: {agent.epsilon:.3f}")
    
    return rewards_hist, recall_hist, duplicates_hist, steps_hist

rewards_hist, recall_hist, dup_hist, steps_hist = train_multi_obj(agent, env, n_episodes=700)

## 5. Training Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
window = 30

# Recall
smoothed_recall = np.convolve(recall_hist, np.ones(window)/window, mode='valid')
axes[0, 0].plot(smoothed_recall, color='green', linewidth=1.5)
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Recall')
axes[0, 0].set_title('Detection Recall')
axes[0, 0].set_ylim(0, 1.05)
axes[0, 0].grid(True, alpha=0.3)

# Duplicates
smoothed_dup = np.convolve(dup_hist, np.ones(window)/window, mode='valid')
axes[0, 1].plot(smoothed_dup, color='red', linewidth=1.5)
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Avg Duplicates')
axes[0, 1].set_title('Duplicate Detections (lower = better)')
axes[0, 1].grid(True, alpha=0.3)

# Rewards
smoothed_r = np.convolve(rewards_hist, np.ones(window)/window, mode='valid')
axes[1, 0].plot(smoothed_r, color='blue', linewidth=1.5)
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Total Reward')
axes[1, 0].set_title('Episode Rewards')
axes[1, 0].grid(True, alpha=0.3)

# Steps
smoothed_s = np.convolve(steps_hist, np.ones(window)/window, mode='valid')
axes[1, 1].plot(smoothed_s, color='orange', linewidth=1.5)
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('Steps')
axes[1, 1].set_title('Steps Per Episode')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('multi_obj_training.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Visualization: Sequential Detection

In [ ]:
def visualize_sequential_detection(agent, env, n_examples=4):
    """Visualize how agent detects objects sequentially."""
    for ex in range(n_examples):
        state = env.reset()
        done = False
        
        while not done:
            action = agent.select_action(state, training=False)
            state, reward, done, info = env.step(action)
        
        # Visualize
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        # Full scene with ground truth
        ax = axes[0]
        ax.imshow(env.image, cmap='gray', vmin=0, vmax=1)
        for i, obj in enumerate(env.objects):
            x, y, w, h = obj['bbox']
            color = 'lime' if obj['detected'] else 'red'
            rect = patches.Rectangle((x, y), w, h, linewidth=2,
                                    edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x, y-2, f'GT{i+1}', fontsize=8, color=color)
        ax.set_title(f'Ground Truth ({len(env.objects)} objects)')
        ax.axis('off')
        
        # Detection sequence
        ax = axes[1]
        ax.imshow(env.image, cmap='gray', vmin=0, vmax=1)
        det_colors = {'correct': 'cyan', 'duplicate': 'red', 'false_positive': 'yellow'}
        for i, det in enumerate(env.detections):
            x, y, w, h = det['bbox']
            color = det_colors[det['type']]
            rect = patches.Rectangle((x, y), w, h, linewidth=1.5,
                                    edgecolor=color, facecolor=color, alpha=0.3)
            ax.add_patch(rect)
            ax.text(x+w/2, y+h/2, str(det['step']), fontsize=8,
                   ha='center', va='center', color='white', fontweight='bold')
        ax.set_title(f'Detections (step order)\nRecall: {info["recall"]:.0%}')
        ax.axis('off')
        
        # Detection mask
        ax = axes[2]
        combined = np.stack([env.image, env.image, env.image], axis=-1)
        mask_overlay = np.zeros_like(combined)
        mask_overlay[:,:,1] = env.detection_mask * 0.5
        ax.imshow(combined + mask_overlay, vmin=0, vmax=1.5)
        ax.set_title(f'Detection Mask\n{info["n_detected"]}/{info["n_objects"]} found')
        ax.axis('off')
        
        plt.suptitle(f'Example {ex+1}: Sequential Multi-Object Detection', fontsize=12)
        plt.tight_layout()
        plt.show()

visualize_sequential_detection(agent, env)

## 7. Performance vs Number of Objects

In [ ]:
def evaluate_by_object_count(agent, n_trials=50):
    """Evaluate agent performance grouped by number of objects."""
    results = {k: {'recall': [], 'duplicates': [], 'steps': []} 
               for k in range(2, 6)}
    
    for n_obj in range(2, 6):
        test_env = MultiObjectEnv(min_objects=n_obj, max_objects=n_obj)
        
        for _ in range(n_trials):
            state = test_env.reset()
            done = False
            n_dup = 0
            
            while not done:
                action = agent.select_action(state, training=False)
                state, reward, done, info = test_env.step(action)
                if reward == -1.0:
                    n_dup += 1
            
            results[n_obj]['recall'].append(info['recall'])
            results[n_obj]['duplicates'].append(n_dup)
            results[n_obj]['steps'].append(info['steps'])
    
    return results

results = evaluate_by_object_count(agent)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
n_objects_range = list(range(2, 6))

# Recall vs num objects
avg_recalls = [np.mean(results[k]['recall']) for k in n_objects_range]
std_recalls = [np.std(results[k]['recall']) for k in n_objects_range]
axes[0].bar(n_objects_range, avg_recalls, yerr=std_recalls, 
           color='green', edgecolor='black', alpha=0.7, capsize=5)
axes[0].set_xlabel('Number of Objects')
axes[0].set_ylabel('Recall')
axes[0].set_title('Detection Recall vs Object Count')
axes[0].set_ylim(0, 1.1)
axes[0].grid(True, alpha=0.3, axis='y')

# Duplicates vs num objects
avg_dups = [np.mean(results[k]['duplicates']) for k in n_objects_range]
axes[1].bar(n_objects_range, avg_dups, color='red', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Number of Objects')
axes[1].set_ylabel('Avg Duplicates')
axes[1].set_title('Duplicate Detections vs Object Count')
axes[1].grid(True, alpha=0.3, axis='y')

# Steps vs num objects
avg_steps = [np.mean(results[k]['steps']) for k in n_objects_range]
axes[2].bar(n_objects_range, avg_steps, color='blue', edgecolor='black', alpha=0.7)
axes[2].set_xlabel('Number of Objects')
axes[2].set_ylabel('Avg Steps')
axes[2].set_title('Steps Used vs Object Count')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('performance_vs_objects.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nPerformance Summary:")
print(f"{'Objects':<10} {'Recall':<12} {'Duplicates':<12} {'Steps':<10}")
print("-" * 44)
for k in n_objects_range:
    print(f"{k:<10} {np.mean(results[k]['recall']):<12.2%} "
          f"{np.mean(results[k]['duplicates']):<12.2f} "
          f"{np.mean(results[k]['steps']):<10.1f}")

## Summary

### Key Takeaways

1. **Sequential Detection:** Multi-object detection can be decomposed into a sequence of single-object detection decisions, with the state tracking what has been found.

2. **Detection Mask as Memory:** The binary mask of already-detected regions serves as a spatial memory that prevents duplicate detections — an RL-based alternative to non-maximum suppression (NMS).

3. **Stopping Policy:** The agent must learn when to stop — triggering too early misses objects, too late wastes computation. The `stop` action with appropriate reward shapes this behavior.

4. **Scalability:** Performance degrades gracefully with more objects, showing the challenge of the credit assignment problem in longer sequences.

### Formal Analysis

The detection recall at convergence follows:

$$\text{Recall}(K) = \mathbb{P}\left[\bigcap_{k=1}^K \text{detect}(o_k)\right] \leq \left(\mathbb{P}[\text{detect}(o_1)]\right)^K$$

with equality when detections are independent. This explains the recall drop with more objects.

The optimal number of detection steps satisfies:

$$T^* = \arg\min_T \left\{\text{miss\_penalty} \cdot (K - \mathbb{E}[n_T]) + \text{dup\_penalty} \cdot \mathbb{E}[d_T]\right\}$$

where $n_T$ is objects found and $d_T$ is duplicates after $T$ steps.

### Connections to Modern Detection
- **DETR:** Uses set-based detection with learned object queries (similar to our grid actions)
- **Autoregressive Detection:** Sequential token prediction for boxes
- **RL-NMS:** Using RL to replace hand-crafted NMS thresholds

### Next Steps
- Continuous action space for precise box localization
- Attention over already-detected objects for relational reasoning
- Curriculum learning: start with fewer objects, gradually increase